In [52]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
import glob
import tqdm
plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 2})

In [53]:
label_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/label/Baeklab.070.GP3.depth5_None.twm6astrict.tsv", sep="\t")
label_df = label_df[["id", "depth", "label", "m6A_level", "5mer", "drach"]]
label_df.rename(columns = {"id": "label_id","m6A_level": "dom_label"}, inplace = True)
label_df.reset_index(inplace=True)
label_df["index"] = label_df.index.astype(np.int32)


In [54]:
logsum_df = "/extdata4/baeklab/Hyeonseo/m6A/inference/inference/AIRNA-DW-v4-20240905-092858-10-177000-token_normalise_dwell_bq_npz_allmotif_pileup_psum/pileup.npz"
with np.load(logsum_df, allow_pickle=True) as data:
    logsum_df = pd.DataFrame({key: data[key] for key in data.keys()})
logsum_df = logsum_df.merge(label_df, on="label_id", how="left")
print(logsum_df)

                label_id     p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
0         NM_000016:1000  0.606511           0.0             0.0          0   
1         NM_000016:1008  0.948786           0.0             0.0          0   
2         NM_000016:1009  2.607620           0.0             0.0          0   
3         NM_000016:1010  4.912612           0.0             0.0          0   
4         NM_000016:1014  3.237954           0.0             0.0          0   
...                  ...       ...           ...             ...        ...   
30894445   NR_184306:979  0.653140           0.0             0.0          0   
30894446    NR_184306:98  4.046034           0.0             0.0          0   
30894447   NR_184306:984  0.009918           0.0             0.0          0   
30894448   NR_184306:986  0.010472           0.0             0.0          0   
30894449   NR_184306:997  0.307928           0.0             0.0          0   

          logsum_p_neg  logsum_1_p_neg  count_neg  

In [55]:
logsum_df["dom_threshold"] = np.log10(0.02)*(logsum_df["dom"]) > logsum_df["logsum_1_p_pos"]/logsum_df["count_all"]
print(logsum_df)

                label_id     p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
0         NM_000016:1000  0.606511           0.0             0.0          0   
1         NM_000016:1008  0.948786           0.0             0.0          0   
2         NM_000016:1009  2.607620           0.0             0.0          0   
3         NM_000016:1010  4.912612           0.0             0.0          0   
4         NM_000016:1014  3.237954           0.0             0.0          0   
...                  ...       ...           ...             ...        ...   
30894445   NR_184306:979  0.653140           0.0             0.0          0   
30894446    NR_184306:98  4.046034           0.0             0.0          0   
30894447   NR_184306:984  0.009918           0.0             0.0          0   
30894448   NR_184306:986  0.010472           0.0             0.0          0   
30894449   NR_184306:997  0.307928           0.0             0.0          0   

          logsum_p_neg  logsum_1_p_neg  count_neg  

In [56]:
print(logsum_df[logsum_df["dom_threshold"] == True])

                label_id      p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
25        NM_000016:1071  13.440902     -0.008027      -36.169472         11   
69        NM_000016:1218   3.342309     -0.013035       -6.126953          3   
76        NM_000016:1232   1.903287     -0.002427       -2.254001          1   
81        NM_000016:1244   4.040272     -0.000032       -4.128505          1   
82        NM_000016:1245   4.232924     -0.001182       -5.788179          2   
...                  ...        ...           ...             ...        ...   
30894396   NR_184306:731   5.942105     -0.004176      -14.097122          5   
30894400   NR_184306:740   2.026625     -0.000042       -4.016816          1   
30894403   NR_184306:769  11.136024     -0.007305      -26.958271          9   
30894404    NR_184306:77  13.487498     -0.001975      -36.067211         10   
30894435    NR_184306:93  32.341377     -0.011076      -78.301079         23   

          logsum_p_neg  logsum_1_p_neg 

In [57]:
from scipy.stats import pearsonr
selected_df = logsum_df[logsum_df["dom_threshold"] == True]
print(pearsonr(selected_df["dom_label"], selected_df["dom"])[0]**2)


0.7051298784003524


In [12]:
print(logsum_df[logsum_df["dom_threshold"] & (logsum_df["count_dom"] <= 100)])

                label_id      p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
389       NM_000016:2072   1.407804     -0.001484       -2.467172          1   
398       NM_000016:2103   2.165038     -0.003002       -4.992980          2   
401       NM_000016:2112   1.101602     -0.007547       -1.763757          1   
883       NM_000017:1825  12.330129     -0.012057      -20.549452          7   
1345      NM_000018:2175   1.569643     -0.000744       -2.766762          1   
...                  ...        ...           ...             ...        ...   
30894396   NR_184306:731   5.942105     -0.004176      -14.097122          5   
30894400   NR_184306:740   2.026625     -0.000042       -4.016816          1   
30894403   NR_184306:769  11.136024     -0.007305      -26.958271          9   
30894404    NR_184306:77  13.487498     -0.001975      -36.067211         10   
30894435    NR_184306:93  32.341377     -0.011076      -78.301079         23   

          logsum_p_neg  logsum_1_p_neg 

In [15]:
print(logsum_df[logsum_df["dom_threshold"] & (logsum_df["count_all"] <= 100)]["count_all"].sum())

24160785


In [16]:
print(logsum_df.columns.tolist())

['label_id', 'p_sum', 'logsum_p_pos', 'logsum_1_p_pos', 'count_pos', 'logsum_p_neg', 'logsum_1_p_neg', 'count_neg', 'logsum_p_all', 'logsum_1_p_all', 'count_all', 'dom', 'count_dom', 'index', 'depth', 'label', 'dom_label', '5mer', 'drach', 'dom_threshold']


In [58]:

def calc_pm6a(df):
    return -(2-df["dom"])*df["logsum_1_p_pos"]/df["count_all"] + ((1-df["dom"])*np.log10(np.clip(1-df["dom"],1e-30,1)) + df["dom"] * np.log10(np.clip(df["dom"],1e-30,1)))*(df["count_pos"]/df["count_all"])

logsum_df["m6a_score"] = calc_pm6a(logsum_df)

In [64]:
postprocess_df = logsum_df[logsum_df["dom_threshold"] & (logsum_df["count_all"] > 100) & (logsum_df["count_all"] <= 200)][['label_id', 'count_all', 'count_dom', 'dom', 'm6a_score', 'label', 'dom_label', '5mer', 'drach']].copy()

In [65]:
postprocess_df["label_index"] = postprocess_df.index
postprocess_df.reset_index(inplace=True, drop=True)

In [66]:
print(postprocess_df)

              label_id  count_all  count_dom       dom  m6a_score  label  \
0         NM_000016:21        129        127  0.007874   0.040554      0   
1       NM_000017:1293        130        126  0.007937   0.045342      0   
2       NM_000017:1310        131        123  0.170732   0.668691     -1   
3       NM_000017:1321        131        126  0.492063   2.021584     -1   
4       NM_000017:1373        130        125  0.632000   2.437110      1   
...                ...        ...        ...       ...        ...    ...   
101896  NR_182661:1835        112        103  0.300971   1.056901      1   
101897  NR_182661:1872        112        110  0.127273   0.517969     -1   
101898   NR_182661:206        110        107  0.009346   0.038448      0   
101899   NR_182661:519        109        108  0.009259   0.040986      0   
101900   NR_182661:566        113        112  0.008929   0.053084     -1   

        dom_label   5mer  drach  label_index  
0        0.000000  GGAGT  False         

In [67]:
postprocess_df.to_csv("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/label/102124_postprocess_label.tsv", sep="\t", index=False)
postprocess_df.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/label/102124_postprocess_label.pkl")

In [32]:
postprocess_df["count_all"].sum()

24683439

In [34]:
postprocess_df["label_id"].nunique()

776815

In [35]:
postprocess_df["label_index"].nunique()

776815

In [37]:
postprocess_df["count_all"].min()

1

In [40]:
postprocess_df_strict = postprocess_df[(postprocess_df["count_all"] <= 100) & (postprocess_df["count_all"] >= 5)]

In [41]:
print(postprocess_df_strict)

              label_id  count_all  count_dom       dom  m6a_score  label  \
0       NM_000016:2072         45         44  0.022727   0.107359      0   
1       NM_000016:2103         36         36  0.055556   0.264506      0   
2       NM_000016:2112         36         36  0.027778   0.095094      0   
3       NM_000017:1825         93         89  0.123596   0.402387      0   
4       NM_000018:2175         77         77  0.012987   0.071006      0   
...                ...        ...        ...       ...        ...    ...   
776810   NR_184306:731         16         16  0.375000   1.341954     -3   
776811   NR_184306:740         15         13  0.076923   0.507125     -3   
776812   NR_184306:769         16         14  0.785714   1.919012     -3   
776813    NR_184306:77         49         46  0.260870   1.229243     -3   
776814    NR_184306:93         50         46  0.695652   1.919874     -3   

        dom_label   5mer  drach  label_index  
0             0.0  TTATT  False         

In [49]:
from scipy import stats
postprocess_df
print(stats.pearsonr(postprocess_df_strict["dom_label"], postprocess_df_strict["dom"])[0]**2)

0.6870098635314965


In [ ]:
orint()

In [48]:
from scipy import stats
postprocess_df_2 = postprocess_df[(postprocess_df["count_all"] <= 100) & (postprocess_df["count_all"] >= 5)]
print(stats.pearsonr(postprocess_df_2["dom"], postprocess_df_2["dom_label"], alternative="greater"))

PearsonRResult(statistic=0.8288605814800801, pvalue=0.0)


In [50]:
print(logsum_df)

                label_id     p_sum  logsum_p_pos  logsum_1_p_pos  count_pos  \
0         NM_000016:1000  0.606511           0.0             0.0          0   
1         NM_000016:1008  0.948786           0.0             0.0          0   
2         NM_000016:1009  2.607620           0.0             0.0          0   
3         NM_000016:1010  4.912612           0.0             0.0          0   
4         NM_000016:1014  3.237954           0.0             0.0          0   
...                  ...       ...           ...             ...        ...   
30894445   NR_184306:979  0.653140           0.0             0.0          0   
30894446    NR_184306:98  4.046034           0.0             0.0          0   
30894447   NR_184306:984  0.009918           0.0             0.0          0   
30894448   NR_184306:986  0.010472           0.0             0.0          0   
30894449   NR_184306:997  0.307928           0.0             0.0          0   

          logsum_p_neg  logsum_1_p_neg  count_neg  

In [51]:
logsum_df.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/label/101424_scatter_df.pkl")

In [ ]:
logsum_df.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/label/101424_scatter_df.pkl")